In [1]:
!pip install -q aif360 fairlearn openpyxl pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 34.6 MB/s eta 0:00:00


In [ ]:
# -*- coding: utf-8 -*-
"""projeto_ml_2va.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Y5RLC6JVUIaK8-qO-gYz2S_8ios9yqL-
"""


from __future__ import annotations

import json
import os
import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

# -------------------------------
# Configurações
# -------------------------------
DATASET_NAMES = [
    "gs_v0_5",
    "gs_v0_25",
    "gs_v0_75",
    "gs_v0",
    "gs_v1",
    "pd_v0_5",
    "pd_v0_25",
    "pd_v0_75",
    "pd_v0",
    "pd_v1",
]

RANDOM_STATE = 42
DEFAULT_COLS = {
    "target": "I-d",
    "protected": "RAMO_ATIVIDADE_1",
    "date": "DATA_LANCAMENTO",
    "account": "NUMERO_CONTA",
    "counterpart_account": "NUMERO_CONTA_OD",
    "counterpart_id": "CPF_CNPJ_OD",
    "holder_id": "CPF_CNPJ_TITULAR",
    "value": "VALOR_TRANSACAO",
    "balance": "VALOR_SALDO",
    "transaction_type": "NATUREZA_LANCAMENTO",
    "cnab": "CNAB",
}

PRIVILEGED_VALUE = 1
UNPRIVILEGED_VALUE = 4

FAST_MODE = os.environ.get("FAST_MODE", "1") == "1"
USE_CACHE = os.environ.get("USE_CACHE", "1") == "1"
PR_VALIDATION_MAX_ROWS = int(os.environ.get("PR_VALIDATION_MAX_ROWS", "30000"))
MAX_PR_TRAIN_ROWS = int(os.environ.get("MAX_PR_TRAIN_ROWS", "15000"))
CACHE_VERSION = "v2"

if FAST_MODE:
    ETA_GRID = [0.0, 1.0, 10.0, 50.0]
    OHE_MIN_FREQUENCY = 0.01
    LR_MAX_ITER = 300
else:
    ETA_GRID = [0.0, 0.5, 1.0, 5.0, 10.0, 25.0, 50.0, 100.0]
    OHE_MIN_FREQUENCY = None
    LR_MAX_ITER = 1000

F1_DROP_TOLERANCE = 0.15  # Flexibiliza o critério de seleção de eta, permitindo encontrar uma penalidade de justiça válida com mais facilidade ao assumir um trade-off maior entre acurácia e justiça.

# -------------------------------
# Utilidades
# -------------------------------
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def find_existing_file(data_dir: str, base_name: str) -> Optional[str]:
    for ext in [".csv", ".xlsx", ".xls", ".parquet"]:
        p = Path(data_dir) / f"{base_name}{ext}"
        if p.exists():
            return str(p)
    return None


def read_table(path: str) -> pd.DataFrame:
    lower = path.lower()
    if lower.endswith(".csv"):
        for kwargs in [
            {"low_memory": False},
            {"sep": ";", "low_memory": False},
            {"sep": ";", "encoding": "latin1", "low_memory": False},
            {"encoding": "latin1", "low_memory": False},
        ]:
            try:
                return pd.read_csv(path, **kwargs)
            except Exception:
                pass
        raise ValueError(f"Falha ao ler CSV: {path}")
    if lower.endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    if lower.endswith(".parquet"):
        return pd.read_parquet(path)
    raise ValueError(f"Formato não suportado: {path}")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [re.sub(r"\s+", "_", str(c).strip()) for c in out.columns]
    return out


def safe_to_datetime(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")


def clean_string_id(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .replace({"nan": np.nan, "None": np.nan, "": np.nan, "<NA>": np.nan})
    )


def safe_binary_target(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.astype("int8")
    if pd.api.types.is_numeric_dtype(s):
        vals = pd.to_numeric(s, errors="coerce")
        unique = set(vals.dropna().unique().tolist())
        if unique.issubset({0, 1}):
            return vals.fillna(0).astype("int8")
        return (vals > 0).fillna(0).astype("int8")
    lowered = s.astype(str).str.lower().str.strip()
    pos = lowered.isin({"1", "true", "sim", "yes", "y", "fraude", "positivo"})
    return pos.astype("int8")


def infer_bias_numeric(name: str) -> float:
    name = name.lower()
    if "0_25" in name:
        return 0.25
    if "0_5" in name:
        return 0.50
    if "0_75" in name:
        return 0.75
    if name.endswith("v0"):
        return 0.0
    if name.endswith("v1"):
        return 1.0
    return np.nan


def bias_bucket(x: float) -> str:
    if pd.isna(x):
        return "desconhecido"
    if x <= 0.25:
        return "baixo"
    if x <= 0.75:
        return "medio"
    return "alto"


def _safe_div(a: float, b: float) -> float:
    return float(a / b) if b not in (0, 0.0) else np.nan


def choose_stratify_series(df: pd.DataFrame, target: str, protected: Optional[str] = None) -> Optional[pd.Series]:
    if protected is not None:
        joint = df[target].astype(str) + "_" + df[protected].astype(str)
        counts = joint.value_counts(dropna=False)
        if joint.nunique() > 1 and (counts >= 2).all():
            return joint

    y = pd.to_numeric(df[target], errors="coerce")
    counts_y = y.value_counts(dropna=False)
    if y.nunique() > 1 and (counts_y >= 2).all():
        return y

    return None


# -------------------------------
# Métricas de fairness
# -------------------------------
def representation_bias(df: pd.DataFrame, protected_col: str) -> Dict[str, float]:
    sub = df[df[protected_col].isin([PRIVILEGED_VALUE, UNPRIVILEGED_VALUE])].copy()
    n_priv = int((sub[protected_col] == PRIVILEGED_VALUE).sum())
    n_unpriv = int((sub[protected_col] == UNPRIVILEGED_VALUE).sum())
    ci = _safe_div(n_priv - n_unpriv, n_priv + n_unpriv)
    return {
        "n_privileged": n_priv,
        "n_unprivileged": n_unpriv,
        "class_imbalance": ci,
        "abs_class_imbalance": abs(ci) if not pd.isna(ci) else np.nan,
    }


def fairness_metrics(y_true: np.ndarray, y_pred: np.ndarray, protected: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    protected = np.asarray(protected)
    mask_p = protected == PRIVILEGED_VALUE
    mask_u = protected == UNPRIVILEGED_VALUE

    if mask_p.sum() == 0 or mask_u.sum() == 0:
        return {
            "spd": np.nan,
            "eod": np.nan,
            "aaod": np.nan,
            "di": np.nan,
            "ppr_priv": np.nan,
            "ppr_unpriv": np.nan,
            "tpr_priv": np.nan,
            "tpr_unpriv": np.nan,
            "fpr_priv": np.nan,
            "fpr_unpriv": np.nan,
        }

    def ppr(mask: np.ndarray) -> float:
        return float(np.mean(y_pred[mask] == 1)) if mask.sum() else np.nan

    def tpr(mask: np.ndarray) -> float:
        pos = y_true[mask] == 1
        return float(np.mean(y_pred[mask][pos] == 1)) if pos.sum() else np.nan

    def fpr(mask: np.ndarray) -> float:
        neg = y_true[mask] == 0
        return float(np.mean(y_pred[mask][neg] == 1)) if neg.sum() else np.nan

    ppr_p, ppr_u = ppr(mask_p), ppr(mask_u)
    tpr_p, tpr_u = tpr(mask_p), tpr(mask_u)
    fpr_p, fpr_u = fpr(mask_p), fpr(mask_u)
    return {
        "spd": ppr_u - ppr_p,
        "eod": tpr_u - tpr_p,
        "aaod": np.nanmean([abs(fpr_u - fpr_p), abs(tpr_u - tpr_p)]),
        "di": _safe_div(ppr_u, ppr_p),
        "ppr_priv": ppr_p,
        "ppr_unpriv": ppr_u,
        "tpr_priv": tpr_p,
        "tpr_unpriv": tpr_u,
        "fpr_priv": fpr_p,
        "fpr_unpriv": fpr_u,
    }


def fairness_penalty(spd: float, eod: float, aaod: float, di: float) -> float:
    vals = [abs(spd), abs(eod), aaod, abs(1 - di) if pd.notna(di) else np.nan]
    vals = [float(v) for v in vals if pd.notna(v)]
    return float(np.mean(vals)) if vals else np.nan


# -------------------------------
# Carregamento e limpeza
# -------------------------------
def detect_columns(df: pd.DataFrame) -> Dict[str, str]:
    cols = set(df.columns)
    detected = DEFAULT_COLS.copy()
    alt_candidates = {
        "protected": ["RAMO_ATIVIDADE_1", "RAMO_ATIVIDADE"],
        "date": ["DATA_LANCAMENTO", "DATA", "DT_LANCAMENTO"],
        "account": ["NUMERO_CONTA", "CONTA", "NR_CONTA"],
        "counterpart_account": ["NUMERO_CONTA_OD", "CONTA_OD"],
        "counterpart_id": ["CPF_CNPJ_OD", "CNPJ_OD", "CPF_CNPJ_DESTINO"],
        "holder_id": ["CPF_CNPJ_TITULAR", "CPF_CNPJ", "CLIENTE_ID"],
        "value": ["VALOR_TRANSACAO", "VALOR", "VL_TRANSACAO"],
        "balance": ["VALOR_SALDO", "SALDO", "VL_SALDO"],
        "transaction_type": ["NATUREZA_LANCAMENTO", "TIPO_TRANSACAO", "NATUREZA"],
        "cnab": ["CNAB"],
        "target": ["I-d", "ID", "FRAUDE", "TARGET", "ALVO"],
    }
    for key, candidates in alt_candidates.items():
        for c in candidates:
            if c in cols:
                detected[key] = c
                break
    return detected


def clean_base(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_columns(df)
    cols = detect_columns(df)

    if cols["date"] in df.columns:
        df[cols["date"]] = safe_to_datetime(df[cols["date"]])
    else:
        raise ValueError("Coluna de data não encontrada.")

    for key in ["account", "counterpart_account", "counterpart_id", "holder_id"]:
        c = cols.get(key)
        if c in df.columns:
            df[c] = clean_string_id(df[c])

    for key in ["value", "balance", "cnab", "protected"]:
        c = cols.get(key)
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if cols["transaction_type"] in df.columns:
        df[cols["transaction_type"]] = df[cols["transaction_type"]].astype(str).str.strip()

    df[cols["target"]] = safe_binary_target(df[cols["target"]])

    df = df.sort_values([cols["account"], cols["date"]]).reset_index(drop=True)
    return df


# -------------------------------
# Feature engineering otimizado
# -------------------------------
def cumulative_nunique(series: pd.Series) -> pd.Series:
    seen = set()
    out = np.empty(len(series), dtype=np.float32)
    for i, x in enumerate(series.values):
        out[i] = len(seen)
        if pd.notna(x):
            seen.add(x)
    return pd.Series(out, index=series.index)


def build_time_features(df: pd.DataFrame, cols: Dict[str, str]) -> pd.DataFrame:
    out = df.copy()
    date_col = cols["date"]
    out["tx_date"] = out[date_col].dt.date
    out["year"] = out[date_col].dt.year.astype("int16")
    out["month"] = out[date_col].dt.month.astype("int8")
    out["day"] = out[date_col].dt.day.astype("int8")
    out["dayofweek"] = out[date_col].dt.dayofweek.astype("int8")
    out["is_weekend"] = out["dayofweek"].isin([5, 6]).astype("int8")
    out["month_period"] = out[date_col].dt.to_period("M").astype(str)
    return out


def build_historical_features(df: pd.DataFrame, cols: Dict[str, str]) -> pd.DataFrame:
    out = build_time_features(df, cols)
    acc = cols["account"]
    date_col = cols["date"]
    value_col = cols["value"]
    balance_col = cols["balance"]
    cp_id = cols["counterpart_id"]
    ttype = cols["transaction_type"]

    out = out.sort_values([acc, date_col]).reset_index(drop=True)
    g = out.groupby(acc, sort=False)

    out["acct_txn_count_prev"] = g.cumcount().astype("int32")
    out["acct_total_value_prev"] = (
        g[value_col].cumsum()
        .groupby(out[acc], sort=False)
        .shift(1)
        .fillna(0)
        .astype("float32")
    )
    out["acct_avg_value_prev"] = np.where(
        out["acct_txn_count_prev"] > 0,
        out["acct_total_value_prev"] / out["acct_txn_count_prev"].replace(0, np.nan),
        0.0,
    ).astype("float32")

    out["acct_balance_sum_prev"] = (
        g[balance_col].cumsum()
        .groupby(out[acc], sort=False)
        .shift(1)
        .fillna(0)
        .astype("float32")
    )
    out["acct_balance_avg_prev"] = np.where(
        out["acct_txn_count_prev"] > 0,
        out["acct_balance_sum_prev"] / out["acct_txn_count_prev"].replace(0, np.nan),
        0.0,
    ).astype("float32")

    out["days_since_prev_txn"] = (
        g[date_col].diff().dt.total_seconds().div(86400).fillna(-1).astype("float32")
    )

    if cp_id in out.columns:
        out["acct_distinct_counterpart_prev"] = (
            g[cp_id].transform(cumulative_nunique).fillna(0).astype("float32")
        )
    else:
        out["acct_distinct_counterpart_prev"] = np.float32(0)

    out["acct_active_days_prev"] = (
        g["tx_date"].transform(cumulative_nunique).fillna(0).astype("float32")
    )

    if ttype in out.columns:
        out["is_credit_like"] = (
            out[ttype].astype(str).str.contains("C|CR|CRED", case=False, regex=True).fillna(False).astype("int8")
        )
        out["acct_credit_count_prev"] = (
            g["is_credit_like"].cumsum()
            .groupby(out[acc], sort=False)
            .shift(1)
            .fillna(0)
            .astype("float32")
        )
        out["acct_credit_ratio_prev"] = np.where(
            out["acct_txn_count_prev"] > 0,
            out["acct_credit_count_prev"] / out["acct_txn_count_prev"].replace(0, np.nan),
            0.0,
        ).astype("float32")
    else:
        out["is_credit_like"] = np.int8(0)
        out["acct_credit_ratio_prev"] = np.float32(0)

    daily_aggs = {"daily_value_sum": (value_col, "sum"), "daily_txn_count": (value_col, "size")}
    if cp_id in out.columns:
        daily_aggs["daily_distinct_cp"] = (cp_id, pd.Series.nunique)
    else:
        daily_aggs["daily_distinct_cp"] = (value_col, "size")

    daily = (
        out.groupby([acc, "tx_date"], as_index=False)
        .agg(**daily_aggs)
        .sort_values([acc, "tx_date"])
    )
    daily["prev_day_value_sum"] = daily.groupby(acc)["daily_value_sum"].shift(1).fillna(0).astype("float32")
    daily["prev_day_txn_count"] = daily.groupby(acc)["daily_txn_count"].shift(1).fillna(0).astype("float32")
    daily["prev_day_distinct_cp"] = daily.groupby(acc)["daily_distinct_cp"].shift(1).fillna(0).astype("float32")
    daily["rolling7_prev_day_value_sum"] = (
        daily.groupby(acc)["daily_value_sum"].transform(lambda s: s.shift(1).rolling(7, min_periods=1).sum())
        .fillna(0)
        .astype("float32")
    )
    if not FAST_MODE:
        daily["rolling30_prev_day_value_sum"] = (
            daily.groupby(acc)["daily_value_sum"].transform(lambda s: s.shift(1).rolling(30, min_periods=1).sum())
            .fillna(0)
            .astype("float32")
        )

    merge_cols = [acc, "tx_date", "prev_day_value_sum", "prev_day_txn_count", "prev_day_distinct_cp", "rolling7_prev_day_value_sum"]
    if not FAST_MODE:
        merge_cols.append("rolling30_prev_day_value_sum")
    out = out.merge(daily[merge_cols], on=[acc, "tx_date"], how="left")

    monthly_aggs = {
        "monthly_value_sum": (value_col, "sum"),
        "monthly_txn_count": (value_col, "size"),
        "monthly_active_days": ("tx_date", pd.Series.nunique),
    }
    if cp_id in out.columns:
        monthly_aggs["monthly_distinct_cp"] = (cp_id, pd.Series.nunique)
    else:
        monthly_aggs["monthly_distinct_cp"] = (value_col, "size")

    monthly = (
        out.groupby([acc, "month_period"], as_index=False)
        .agg(**monthly_aggs)
        .sort_values([acc, "month_period"])
    )
    for c in ["monthly_value_sum", "monthly_txn_count", "monthly_active_days", "monthly_distinct_cp"]:
        monthly[f"prev_{c}"] = monthly.groupby(acc)[c].shift(1).fillna(0).astype("float32")
    out = out.merge(
        monthly[[acc, "month_period", "prev_monthly_value_sum", "prev_monthly_txn_count", "prev_monthly_active_days", "prev_monthly_distinct_cp"]],
        on=[acc, "month_period"],
        how="left",
    )

    out["ratio_value_to_hist_avg"] = np.where(
        out["acct_avg_value_prev"] > 0,
        out[value_col] / out["acct_avg_value_prev"],
        0.0,
    ).astype("float32")
    out["value_minus_hist_avg"] = (out[value_col] - out["acct_avg_value_prev"]).astype("float32")
    out["balance_minus_hist_avg"] = (out[balance_col] - out["acct_balance_avg_prev"]).astype("float32")
    out["txn_index_in_day"] = out.groupby([acc, "tx_date"]).cumcount().astype("int16")

    keep_int = {cols["target"], cols["protected"], "is_credit_like", "is_weekend", "year", "month", "day", "dayofweek", "txn_index_in_day", "acct_txn_count_prev"}
    for c in out.select_dtypes(include=["float64"]).columns:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype("float32")
    for c in out.select_dtypes(include=["int64"]).columns:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype("int32" if c in keep_int else "float32")

    return out


def make_feature_sets(df: pd.DataFrame, cols: Dict[str, str]) -> Dict[str, List[str]]:
    target = cols["target"]
    protected = cols["protected"]
    ids = [cols[k] for k in ["account", "counterpart_account", "counterpart_id", "holder_id"] if cols.get(k) in df.columns]

    basic = [
        c for c in [
            cols["value"], cols["balance"], cols["cnab"], cols["transaction_type"],
            "year", "month", "day", "dayofweek", "is_weekend", "txn_index_in_day",
        ] if c in df.columns
    ]

    intermediate = basic + [
        c for c in [
            "acct_txn_count_prev", "acct_total_value_prev", "acct_avg_value_prev",
            "acct_balance_avg_prev", "days_since_prev_txn", "acct_distinct_counterpart_prev",
            "acct_active_days_prev", "acct_credit_ratio_prev",
            "prev_day_value_sum", "prev_day_txn_count", "prev_day_distinct_cp",
        ] if c in df.columns
    ]

    rich = intermediate + [
        c for c in [
            "rolling7_prev_day_value_sum",
            "rolling30_prev_day_value_sum",
            "prev_monthly_value_sum", "prev_monthly_txn_count", "prev_monthly_active_days", "prev_monthly_distinct_cp",
            "ratio_value_to_hist_avg", "value_minus_hist_avg", "balance_minus_hist_avg",
        ] if c in df.columns
    ]

    feature_sets = {
        "A_sem_protegido": [c for c in basic if c not in ids + [target, protected]],
        "B_sem_protegido": [c for c in intermediate if c not in ids + [target, protected]],
        "C_sem_protegido": [c for c in rich if c not in ids + [target, protected]],
        "A_com_protegido": [c for c in basic + [protected] if c not in ids + [target]],
        "B_com_protegido": [c for c in intermediate + [protected] if c not in ids + [target]],
        "C_com_protegido": [c for c in rich + [protected] if c not in ids + [target]],
    }

    for k, v in feature_sets.items():
        feature_sets[k] = [c for c in dict.fromkeys(v) if c in df.columns]

    return feature_sets


# -------------------------------
# Modelagem
# -------------------------------
def infer_feature_types(df: pd.DataFrame, features: List[str]) -> Tuple[List[str], List[str]]:
    numeric, categorical = [], []
    for c in features:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric.append(c)
        else:
            categorical.append(c)
    return numeric, categorical


def build_pipeline(df_train: pd.DataFrame, features: List[str]) -> Pipeline:
    numeric, categorical = infer_feature_types(df_train, features)

    cat_ohe_kwargs = {"handle_unknown": "ignore"}
    if OHE_MIN_FREQUENCY is not None:
        cat_ohe_kwargs["min_frequency"] = OHE_MIN_FREQUENCY

    pre = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(**cat_ohe_kwargs)),
                ]),
                categorical,
            ),
        ],
        remainder="drop",
    )

    model = LogisticRegression(
        max_iter=LR_MAX_ITER,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocess", pre), ("model", model)])


def compute_performance(y_true: np.ndarray, y_pred: np.ndarray, y_score: Optional[np.ndarray]) -> Dict[str, float]:
    res = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": np.nan,
        "pr_auc": np.nan,
    }
    if y_score is not None and len(np.unique(y_true)) > 1:
        try:
            res["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            pass
        try:
            res["pr_auc"] = average_precision_score(y_true, y_score)
        except Exception:
            pass
    return res


def fit_and_evaluate(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    features: List[str],
    train_cols: Dict[str, str],
    test_cols: Dict[str, str],
) -> Dict[str, float]:
    target_train = train_cols["target"]
    target_test = test_cols["target"]
    protected_test = test_cols["protected"]

    pipe = build_pipeline(train_df, features)
    pipe.fit(train_df[features], train_df[target_train])
    y_pred = pipe.predict(test_df[features])
    y_score = pipe.predict_proba(test_df[features])[:, 1] if hasattr(pipe, "predict_proba") else None

    perf = compute_performance(test_df[target_test].values, y_pred, y_score)
    fair = fairness_metrics(test_df[target_test].values, y_pred, test_df[protected_test].values)
    return {**perf, **fair, "fairness_penalty": fairness_penalty(fair["spd"], fair["eod"], fair["aaod"], fair["di"])}


# -------------------------------
# Prejudice Remover
# -------------------------------
def filter_protected_groups(df: pd.DataFrame, protected_col: str) -> pd.DataFrame:
    out = df.copy()
    out[protected_col] = pd.to_numeric(out[protected_col], errors="coerce")
    return out[out[protected_col].isin([PRIVILEGED_VALUE, UNPRIVILEGED_VALUE])].copy()


def has_both_groups(df: pd.DataFrame, protected_col: str) -> bool:
    vals = set(pd.to_numeric(df[protected_col], errors="coerce").dropna().unique().tolist())
    return PRIVILEGED_VALUE in vals and UNPRIVILEGED_VALUE in vals


def sanitize_pr_features(features: List[str], cols: Dict[str, str]) -> List[str]:
    target = cols["target"]
    protected = cols["protected"]
    cleaned = [f for f in features if f not in {target, protected}]
    return list(dict.fromkeys(cleaned))


def fit_aif360_category_maps(df: pd.DataFrame, features: List[str], cols: Dict[str, str]) -> Dict[str, Dict[str, int]]:
    target = cols["target"]
    use_cols = list(dict.fromkeys(features + [target]))
    cat_maps: Dict[str, Dict[str, int]] = {}

    for c in use_cols:
        if c == target:
            continue
        if not pd.api.types.is_numeric_dtype(df[c]):
            values = (
                df[c]
                .astype(str)
                .fillna("__MISSING__")
                .replace({"nan": "__MISSING__", "None": "__MISSING__", "<NA>": "__MISSING__"})
            )
            uniques = pd.Index(values.unique().tolist())
            cat_maps[c] = {v: i for i, v in enumerate(uniques)}
    return cat_maps


def transform_for_aif360(
    df: pd.DataFrame,
    features: List[str],
    cols: Dict[str, str],
    protected_aif_col: str,
    cat_maps: Dict[str, Dict[str, int]],
) -> pd.DataFrame:
    target = cols["target"]
    protected = cols["protected"]
    use_cols = list(dict.fromkeys(features + [target]))
    out = df[use_cols].copy()

    for c in out.columns:
        if c == target:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype("int32")
        elif pd.api.types.is_numeric_dtype(out[c]):
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype("float32")
        else:
            values = (
                out[c]
                .astype(str)
                .fillna("__MISSING__")
                .replace({"nan": "__MISSING__", "None": "__MISSING__", "<NA>": "__MISSING__"})
            )
            mapping = cat_maps.get(c, {})
            out[c] = values.map(mapping).fillna(-1).astype("float32")

    out[protected_aif_col] = np.where(
        pd.to_numeric(df[protected], errors="coerce") == PRIVILEGED_VALUE,
        1,
        0,
    ).astype("int32")
    return out


def extract_aif360_predictions(pred) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    labels = getattr(pred, "labels", None)
    if labels is None:
        raise ValueError("Objeto de predição sem atributo 'labels'.")
    labels = np.asarray(labels)
    if labels.ndim == 2:
        if labels.shape[1] == 1:
            labels = labels[:, 0]
        else:
            labels = labels.ravel()
    y_pred = labels.astype(int)

    scores = getattr(pred, "scores", None)
    if scores is None:
        return y_pred, None

    scores = np.asarray(scores)
    if scores.ndim == 2:
        if scores.shape[1] == 1:
            scores = scores[:, 0]
        else:
            scores = scores[:, -1]
    else:
        scores = scores.ravel()
    return y_pred, scores.astype(float)

def sample_pr_train_dataframe(train_df_pr: pd.DataFrame, target: str, protected: str) -> pd.DataFrame:
    if MAX_PR_TRAIN_ROWS <= 0 or len(train_df_pr) <= MAX_PR_TRAIN_ROWS:
        return train_df_pr

    stratify_used = choose_stratify_series(train_df_pr, target=target, protected=protected)

    # Reduzir o tamanho do treino do Prejudice Remover é uma prática justificável para mitigar
    # o custo computacional e os possíveis efeitos negativos de bases excessivamente grandes na
    # equidade, em linha com a literatura-base do projeto sobre o trade-off entre dados, features e fairness.
    sampled_df, _ = train_test_split(
        train_df_pr,
        train_size=MAX_PR_TRAIN_ROWS,
        random_state=RANDOM_STATE,
        stratify=stratify_used,
    )
    return sampled_df.reset_index(drop=True)


def try_run_prejudice_remover(train_df: pd.DataFrame, valid_df: pd.DataFrame, features: List[str], cols: Dict[str, str], eta: float) -> Dict[str, float]:
    try:
        from aif360.algorithms.inprocessing import PrejudiceRemover
        from aif360.datasets import StandardDataset
    except Exception as e:
        return {"error": f"AIF360 não disponível: {e}"}

    target = cols["target"]
    protected = cols["protected"]
    protected_aif_col = f"{protected}_aif"

    train_df = filter_protected_groups(train_df, protected)
    valid_df = filter_protected_groups(valid_df, protected)

    if train_df.empty or valid_df.empty:
        return {"error": "Conjunto vazio após filtrar grupos protegidos 1 e 4."}
    if not has_both_groups(train_df, protected):
        return {"error": "Treino sem ambos os grupos protegidos (1 e 4)."}
    if not has_both_groups(valid_df, protected):
        return {"error": "Validação/teste sem ambos os grupos protegidos (1 e 4)."}
    if pd.to_numeric(train_df[target], errors="coerce").nunique() < 2:
        return {"error": "Treino com apenas uma classe no target."}
    if pd.to_numeric(valid_df[target], errors="coerce").nunique() < 2:
        return {"error": "Validação/teste com apenas uma classe no target."}

    features_pr = sanitize_pr_features(features, cols)
    if len(features_pr) < 1:
        return {"error": "Nenhuma feature restante após remover target/protected."}

    cat_maps = fit_aif360_category_maps(train_df, features_pr, cols)
    tr_df = transform_for_aif360(train_df, features_pr, cols, protected_aif_col, cat_maps)
    va_df = transform_for_aif360(valid_df, features_pr, cols, protected_aif_col, cat_maps)

    try:
        std_train = StandardDataset(
            tr_df,
            label_name=target,
            favorable_classes=[1],
            protected_attribute_names=[protected_aif_col],
            privileged_classes=[[1]],
        )
        std_valid = StandardDataset(
            va_df,
            label_name=target,
            favorable_classes=[1],
            protected_attribute_names=[protected_aif_col],
            privileged_classes=[[1]],
        )

        model = PrejudiceRemover(eta=float(eta), sensitive_attr=protected_aif_col, class_attr=target)
        model.fit(std_train)
        pred = model.predict(std_valid)
        y_pred, scores = extract_aif360_predictions(pred)
    except Exception as e:
        return {"error": f"Falha no Prejudice Remover: {e}"}

    y_true = valid_df[target].values.astype(int)
    if len(y_pred) != len(y_true):
        return {"error": f"Tamanho inconsistente nas predições do Prejudice Remover: y_true={len(y_true)} y_pred={len(y_pred)}"}

    perf = compute_performance(y_true, y_pred, scores)
    fair = fairness_metrics(y_true, y_pred, valid_df[protected].values)
    return {**perf, **fair, "fairness_penalty": fairness_penalty(fair["spd"], fair["eod"], fair["aaod"], fair["di"])}


def choose_eta_via_validation(train_df: pd.DataFrame, features: List[str], cols: Dict[str, str]) -> Tuple[pd.DataFrame, float]:
    target = cols["target"]
    protected = cols["protected"]

    work_df = filter_protected_groups(train_df, protected)

    if FAST_MODE and PR_VALIDATION_MAX_ROWS > 0 and len(work_df) > PR_VALIDATION_MAX_ROWS:
        work_df = work_df.sample(PR_VALIDATION_MAX_ROWS, random_state=RANDOM_STATE)

    if not has_both_groups(work_df, protected):
        return pd.DataFrame([{
            "eta": 0.0,
            "error": "Base de treino sem ambos os grupos protegidos (1 e 4)."
        }]), 0.0

    stratify_used = choose_stratify_series(work_df, target=target, protected=protected)

    base_train, base_valid = train_test_split(
        work_df,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=stratify_used,
    )

    if not has_both_groups(base_train, protected):
        return pd.DataFrame([{
            "eta": 0.0,
            "error": "Split de treino sem ambos os grupos protegidos."
        }]), 0.0

    if not has_both_groups(base_valid, protected):
        return pd.DataFrame([{
            "eta": 0.0,
            "error": "Split de validação sem ambos os grupos protegidos."
        }]), 0.0

    rows = []
    for eta in ETA_GRID:
        res = try_run_prejudice_remover(base_train, base_valid, features, cols, eta)
        res["eta"] = eta
        rows.append(res)
    val_df = pd.DataFrame(rows)

    if "error" in val_df.columns and val_df["error"].notna().all():
        return val_df, 0.0

    error_col = val_df["error"] if "error" in val_df.columns else pd.Series([None] * len(val_df), index=val_df.index)
    valid_rows = val_df[error_col.isna()].copy()
    if valid_rows.empty:
        return val_df, 0.0

    baseline_f1 = valid_rows.loc[valid_rows["eta"] == 0.0, "f1"]
    baseline_f1 = float(baseline_f1.iloc[0]) if not baseline_f1.empty else float(valid_rows["f1"].max())
    threshold = baseline_f1 * (1.0 - F1_DROP_TOLERANCE)

    feasible = valid_rows[valid_rows["f1"] >= threshold].copy()
    pool = feasible if not feasible.empty else valid_rows
    chosen = pool.sort_values(["fairness_penalty", "eta", "f1"], ascending=[True, True, False]).iloc[0]
    return val_df, float(chosen["eta"])


# -------------------------------
# Execução principal
# -------------------------------
def summarize_dataset(df: pd.DataFrame, name: str, cols: Dict[str, str]) -> Dict[str, float]:
    target = cols["target"]
    protected = cols["protected"]
    rb = representation_bias(df, protected)

    sub = filter_protected_groups(df, protected)
    priv_mask = sub[protected] == PRIVILEGED_VALUE
    unpriv_mask = sub[protected] == UNPRIVILEGED_VALUE

    fraud_rate_privileged = float(sub.loc[priv_mask, target].mean()) if priv_mask.any() else np.nan
    fraud_rate_unprivileged = float(sub.loc[unpriv_mask, target].mean()) if unpriv_mask.any() else np.nan
    positive_label_gap = (
        fraud_rate_unprivileged - fraud_rate_privileged
        if pd.notna(fraud_rate_privileged) and pd.notna(fraud_rate_unprivileged)
        else np.nan
    )
    positive_label_di = _safe_div(fraud_rate_unprivileged, fraud_rate_privileged)

    n_total = len(sub)
    representation_rate_privileged = _safe_div(int(priv_mask.sum()), n_total)
    representation_rate_unprivileged = _safe_div(int(unpriv_mask.sum()), n_total)

    return {
        "dataset": name,
        "rows": len(df),
        "fraud_rate": float(df[target].mean()),
        "bias_level_name": infer_bias_numeric(name),
        "bias_bucket_name": bias_bucket(infer_bias_numeric(name)),
        "representation_rate_privileged": representation_rate_privileged,
        "representation_rate_unprivileged": representation_rate_unprivileged,
        "fraud_rate_privileged": fraud_rate_privileged,
        "fraud_rate_unprivileged": fraud_rate_unprivileged,
        "positive_label_gap": positive_label_gap,
        "positive_label_di": positive_label_di,
        **rb,
    }


def run_cross_dataset_experiment(datasets: Dict[str, pd.DataFrame], feature_sets_map: Dict[str, Dict[str, List[str]]], cols_map: Dict[str, Dict[str, str]]) -> pd.DataFrame:
    rows = []
    names = list(datasets.keys())
    for train_name in names:
        train_df = datasets[train_name]
        train_cols = cols_map[train_name]
        for feature_set_name, train_features in feature_sets_map[train_name].items():
            for test_name in names:
                if train_name == test_name:
                    continue
                test_df = datasets[test_name]
                common_features = [f for f in train_features if f in test_df.columns]
                if len(common_features) < 2:
                    continue
                try:
                    test_cols = cols_map[test_name]
                    res = fit_and_evaluate(train_df, test_df, common_features, train_cols, test_cols)
                except Exception as e:
                    res = {k: np.nan for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "spd", "eod", "aaod", "di", "fairness_penalty"]}
                    res["error"] = str(e)
                rows.append({
                    "train_dataset": train_name,
                    "test_dataset": test_name,
                    "feature_set": feature_set_name,
                    "n_features": len(common_features),
                    "train_bias_level": infer_bias_numeric(train_name),
                    "test_bias_level": infer_bias_numeric(test_name),
                    "train_bias_bucket": bias_bucket(infer_bias_numeric(train_name)),
                    "test_bias_bucket": bias_bucket(infer_bias_numeric(test_name)),
                    **res,
                })
    return pd.DataFrame(rows)


def aggregate_feature_results(results: pd.DataFrame) -> pd.DataFrame:
    g = results.groupby("feature_set", dropna=False)
    agg = g.agg(
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        fairness_penalty_mean=("fairness_penalty", "mean"),
        spd_abs_mean=("spd", lambda s: np.nanmean(np.abs(s))),
        eod_abs_mean=("eod", lambda s: np.nanmean(np.abs(s))),
        aaod_mean=("aaod", "mean"),
        di_distance_mean=("di", lambda s: np.nanmean(np.abs(1 - s))),
        roc_auc_mean=("roc_auc", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
    ).reset_index()
    agg["selection_score"] = agg["f1_mean"] - agg["fairness_penalty_mean"]
    return agg.sort_values(["selection_score", "f1_mean"], ascending=[False, False])


def aggregate_feature_results_by_test_bias(results: pd.DataFrame) -> pd.DataFrame:
    return (
        results.groupby(["feature_set", "test_bias_bucket"], dropna=False)
        .agg(
            f1_mean=("f1", "mean"),
            fairness_penalty_mean=("fairness_penalty", "mean"),
            spd_abs_mean=("spd", lambda s: np.nanmean(np.abs(s))),
            eod_abs_mean=("eod", lambda s: np.nanmean(np.abs(s))),
            aaod_mean=("aaod", "mean"),
            di_distance_mean=("di", lambda s: np.nanmean(np.abs(1 - s))),
            n=("f1", "count"),
        )
        .reset_index()
        .sort_values(["feature_set", "test_bias_bucket"])
    )


def summarize_feature_set_answers(feature_summary: pd.DataFrame, feature_by_bias: pd.DataFrame) -> pd.DataFrame:
    overall_best_fair = (
        feature_summary.sort_values(["fairness_penalty_mean", "f1_mean"], ascending=[True, False])
        .head(1)[["feature_set", "f1_mean", "fairness_penalty_mean"]]
        .assign(answer_focus="melhor_equilibrio_global")
    )
    overall_best_f1 = (
        feature_summary.sort_values(["f1_mean", "fairness_penalty_mean"], ascending=[False, True])
        .head(1)[["feature_set", "f1_mean", "fairness_penalty_mean"]]
        .assign(answer_focus="melhor_desempenho_global")
    )
    by_bias = (
        feature_by_bias.sort_values(["test_bias_bucket", "fairness_penalty_mean", "f1_mean"], ascending=[True, True, False])
        .groupby("test_bias_bucket", as_index=False)
        .first()[["test_bias_bucket", "feature_set", "f1_mean", "fairness_penalty_mean"]]
        .assign(answer_focus="melhor_por_nivel_de_vies")
    )
    return pd.concat([overall_best_fair, overall_best_f1, by_bias], ignore_index=True, sort=False)


def aggregate_pr_results(pr_test_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    valid = pr_test_df.copy()
    if "error" in valid.columns:
        valid = valid[valid["error"].isna()].copy()

    overall = (
        valid.groupby("eta", dropna=False)
        .agg(
            f1_mean=("f1", "mean"),
            fairness_penalty_mean=("fairness_penalty", "mean"),
            spd_abs_mean=("spd", lambda s: np.nanmean(np.abs(s))),
            eod_abs_mean=("eod", lambda s: np.nanmean(np.abs(s))),
            aaod_mean=("aaod", "mean"),
            di_distance_mean=("di", lambda s: np.nanmean(np.abs(1 - s))),
            n=("f1", "count"),
        )
        .reset_index()
        .sort_values("eta")
    )

    by_bias = (
        valid.groupby(["test_bias_bucket", "eta"], dropna=False)
        .agg(
            f1_mean=("f1", "mean"),
            fairness_penalty_mean=("fairness_penalty", "mean"),
            spd_abs_mean=("spd", lambda s: np.nanmean(np.abs(s))),
            eod_abs_mean=("eod", lambda s: np.nanmean(np.abs(s))),
            aaod_mean=("aaod", "mean"),
            di_distance_mean=("di", lambda s: np.nanmean(np.abs(1 - s))),
            n=("f1", "count"),
        )
        .reset_index()
        .sort_values(["test_bias_bucket", "eta"])
    )
    return overall, by_bias


def build_baseline_vs_pr_table(cross_results: pd.DataFrame, pr_test_df: pd.DataFrame, chosen_feature_set: str) -> pd.DataFrame:
    baseline = (
        cross_results.loc[cross_results["feature_set"] == chosen_feature_set]
        .groupby("test_bias_bucket", dropna=False)
        .agg(
            f1_mean=("f1", "mean"),
            fairness_penalty_mean=("fairness_penalty", "mean"),
            spd_abs_mean=("spd", lambda s: np.nanmean(np.abs(s))),
            eod_abs_mean=("eod", lambda s: np.nanmean(np.abs(s))),
            aaod_mean=("aaod", "mean"),
            di_distance_mean=("di", lambda s: np.nanmean(np.abs(1 - s))),
            n=("f1", "count"),
        )
        .reset_index()
    )
    baseline["model_stage"] = "baseline"

    pr_valid = pr_test_df.copy()
    if "error" in pr_valid.columns:
        pr_valid = pr_valid[pr_valid["error"].isna()].copy()
    pr = (
        pr_valid.groupby("test_bias_bucket", dropna=False)
        .agg(
            f1_mean=("f1", "mean"),
            fairness_penalty_mean=("fairness_penalty", "mean"),
            spd_abs_mean=("spd", lambda s: np.nanmean(np.abs(s))),
            eod_abs_mean=("eod", lambda s: np.nanmean(np.abs(s))),
            aaod_mean=("aaod", "mean"),
            di_distance_mean=("di", lambda s: np.nanmean(np.abs(1 - s))),
            n=("f1", "count"),
        )
        .reset_index()
    )
    pr["model_stage"] = "prejudice_remover"

    cols = ["model_stage", "test_bias_bucket", "f1_mean", "fairness_penalty_mean", "spd_abs_mean", "eod_abs_mean", "aaod_mean", "di_distance_mean", "n"]
    return pd.concat([baseline[cols], pr[cols]], ignore_index=True).sort_values(["test_bias_bucket", "model_stage"])


def run_stat_tests(results: pd.DataFrame) -> pd.DataFrame:
    metrics = ["f1", "fairness_penalty", "spd", "eod", "aaod", "di"]
    rows = []
    for m in metrics:
        groups, labels = [], []
        for fs, grp in results.groupby("feature_set"):
            vals = grp[m].dropna().values
            if len(vals) > 0:
                groups.append(vals)
                labels.append(fs)
        if len(groups) >= 2:
            try:
                stat, p = stats.kruskal(*groups)
                rows.append({"metric": m, "test": "kruskal", "statistic": stat, "pvalue": p, "n_groups": len(groups), "groups": ", ".join(labels)})
            except Exception as e:
                rows.append({"metric": m, "test": "kruskal", "statistic": np.nan, "pvalue": np.nan, "n_groups": len(groups), "groups": ", ".join(labels), "error": str(e)})
    return pd.DataFrame(rows)


def run_bias_level_analysis(results: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    by_test = results.groupby(["feature_set", "test_bias_bucket"], dropna=False).agg(
        f1_mean=("f1", "mean"),
        fairness_penalty_mean=("fairness_penalty", "mean"),
        spd_abs_mean=("spd", lambda s: np.nanmean(np.abs(s))),
        eod_abs_mean=("eod", lambda s: np.nanmean(np.abs(s))),
        aaod_mean=("aaod", "mean"),
        di_distance_mean=("di", lambda s: np.nanmean(np.abs(1 - s))),
        n=("f1", "count"),
    ).reset_index()

    by_pair = results.groupby(["feature_set", "train_bias_bucket", "test_bias_bucket"], dropna=False).agg(
        f1_mean=("f1", "mean"),
        fairness_penalty_mean=("fairness_penalty", "mean"),
        n=("f1", "count"),
    ).reset_index()
    return by_test, by_pair


def select_fixed_feature_set(feature_summary: pd.DataFrame) -> str:
    if feature_summary.empty:
        raise RuntimeError("Resumo de feature sets vazio.")
    sem_protegido = feature_summary[feature_summary["feature_set"].str.endswith("_sem_protegido")].copy()
    pool = sem_protegido if not sem_protegido.empty else feature_summary
    return str(pool.sort_values(["selection_score", "f1_mean"], ascending=[False, False]).iloc[0]["feature_set"])


def run_prejudice_remover_experiment(datasets: Dict[str, pd.DataFrame], feature_sets_map: Dict[str, Dict[str, List[str]]], cols_map: Dict[str, Dict[str, str]], chosen_feature_set: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    eta_validation_rows = []
    test_rows = []
    names = list(datasets.keys())

    for train_name in names:
        train_df = datasets[train_name]
        train_cols = cols_map[train_name]
        train_features = feature_sets_map[train_name][chosen_feature_set]
        val_df, best_eta = choose_eta_via_validation(train_df, train_features, train_cols)
        val_df.insert(0, "train_dataset", train_name)
        val_df.insert(1, "feature_set", chosen_feature_set)
        val_df["chosen_eta"] = best_eta
        eta_validation_rows.append(val_df)

        if best_eta == 0.0 and "error" in val_df.columns and val_df["error"].notna().all():
            continue

        train_df_pr = filter_protected_groups(train_df, train_cols["protected"])
        if not has_both_groups(train_df_pr, train_cols["protected"]):
            continue
        train_df_pr = sample_pr_train_dataframe(
            train_df_pr=train_df_pr,
            target=train_cols["target"],
            protected=train_cols["protected"],
        )
        if not has_both_groups(train_df_pr, train_cols["protected"]):
            continue

        for test_name in names:
            if train_name == test_name:
                continue
            test_df = datasets[test_name]
            test_df_pr = filter_protected_groups(test_df, train_cols["protected"])
            if not has_both_groups(test_df_pr, train_cols["protected"]):
                test_rows.append({
                    "train_dataset": train_name,
                    "test_dataset": test_name,
                    "feature_set": chosen_feature_set,
                    "eta": best_eta,
                    "train_bias_level": infer_bias_numeric(train_name),
                    "test_bias_level": infer_bias_numeric(test_name),
                    "train_bias_bucket": bias_bucket(infer_bias_numeric(train_name)),
                    "test_bias_bucket": bias_bucket(infer_bias_numeric(test_name)),
                    "error": "Base de teste sem ambos os grupos protegidos (1 e 4).",
                })
                continue
            common_features = [f for f in train_features if f in test_df_pr.columns]
            common_features = sanitize_pr_features(common_features, train_cols)
            if len(common_features) < 1:
                continue
            try:
                res = try_run_prejudice_remover(train_df_pr, test_df_pr, common_features, train_cols, best_eta)
            except Exception as e:
                res = {"error": str(e)}
            test_rows.append({
                "train_dataset": train_name,
                "test_dataset": test_name,
                "feature_set": chosen_feature_set,
                "eta": best_eta,
                "train_bias_level": infer_bias_numeric(train_name),
                "test_bias_level": infer_bias_numeric(test_name),
                "train_bias_bucket": bias_bucket(infer_bias_numeric(train_name)),
                "test_bias_bucket": bias_bucket(infer_bias_numeric(test_name)),
                **res,
            })

    eta_validation_df = pd.concat(eta_validation_rows, ignore_index=True) if eta_validation_rows else pd.DataFrame()
    test_df = pd.DataFrame(test_rows)
    return eta_validation_df, test_df


# -------------------------------
# Relatórios e gráficos
# -------------------------------
def save_plot_feature_summary(feature_summary: pd.DataFrame, output_dir: str) -> None:
    if feature_summary.empty:
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(feature_summary))
    ax.bar(x - 0.2, feature_summary["f1_mean"], width=0.4, label="F1 médio")
    ax.bar(x + 0.2, feature_summary["fairness_penalty_mean"], width=0.4, label="Penalty fairness")
    ax.set_xticks(x)
    ax.set_xticklabels(feature_summary["feature_set"], rotation=45, ha="right")
    ax.set_title("Comparação entre conjuntos de features")
    ax.legend()
    fig.tight_layout()
    fig.savefig(Path(output_dir) / "feature_sets_f1_vs_fairness.png", dpi=140)
    plt.close(fig)


def save_plot_eta_validation(eta_validation_df: pd.DataFrame, output_dir: str) -> None:
    if eta_validation_df.empty:
        return

    required_cols = {"eta", "f1", "fairness_penalty"}
    if not required_cols.issubset(set(eta_validation_df.columns)):
        print("[WARN] Não foi possível gerar o gráfico de eta: colunas necessárias ausentes.")
        print(f"[WARN] Colunas disponíveis: {list(eta_validation_df.columns)}")
        return

    summary = (
        eta_validation_df.groupby("eta", dropna=False)
        .agg(
            f1_mean=("f1", "mean"),
            fairness_penalty_mean=("fairness_penalty", "mean"),
        )
        .reset_index()
    )

    if summary.empty:
        print("[WARN] Resumo de validação de eta vazio.")
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(summary["eta"], summary["f1_mean"], marker="o", label="F1 médio")
    ax.plot(summary["eta"], summary["fairness_penalty_mean"], marker="o", label="Penalty fairness")
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlabel("eta")
    ax.set_title("Escolha de eta na validação")
    ax.legend()
    fig.tight_layout()
    fig.savefig(Path(output_dir) / "eta_validacao.png", dpi=140)
    plt.close(fig)


def write_text_report(
    output_dir: str,
    dataset_summary: pd.DataFrame,
    feature_summary: pd.DataFrame,
    feature_by_bias: pd.DataFrame,
    feature_answers: pd.DataFrame,
    stat_tests: pd.DataFrame,
    chosen_feature_set: str,
    eta_validation_df: pd.DataFrame,
    pr_overall_summary: pd.DataFrame,
    pr_by_bias_summary: pd.DataFrame,
    baseline_vs_pr: pd.DataFrame,
    pr_test_df: pd.DataFrame,
) -> None:
    path = Path(output_dir) / "relatorio_resumido.txt"
    lines = []
    lines.append("Projeto Tema 2 - Resumo automático")
    lines.append("Interpretação adotada para os 10 ciclos: cada uma das 10 bases é usada uma vez como base de treino e testada nas demais.")
    lines.append(f"FAST_MODE={FAST_MODE}")
    if FAST_MODE:
        lines.append("ATENÇÃO: os resultados foram gerados em FAST_MODE; use execução completa antes da versão final de entrega.")
    lines.append("")
    lines.append("=== Diagnóstico de viés das bases ===")
    lines.append(dataset_summary.to_string(index=False))
    lines.append("")
    lines.append("=== RQ 2.1 - comparação global entre conjuntos de features ===")
    lines.append(feature_summary.to_string(index=False))
    lines.append("")
    lines.append("=== RQ 2.1 - comparação por nível de viés da base de teste ===")
    lines.append(feature_by_bias.to_string(index=False))
    lines.append("")
    lines.append("=== RQ 2.1 - síntese pronta para discussão ===")
    lines.append(feature_answers.to_string(index=False))
    lines.append("")
    lines.append("=== RQ 2.1 - testes estatísticos entre feature sets ===")
    lines.append(stat_tests.to_string(index=False) if not stat_tests.empty else "Sem testes disponíveis.")
    lines.append("")
    lines.append("Critério de seleção do feature set fixo: maior (F1 médio - fairness_penalty médio) entre os conjuntos sem atributo protegido.")
    lines.append(f"Feature set fixo escolhido para RQ 2.2: {chosen_feature_set}")
    if not eta_validation_df.empty:
        chosen_eta_summary = eta_validation_df.groupby("train_dataset")["chosen_eta"].first().reset_index()
        lines.append("")
        lines.append("=== Eta escolhido por base de treino ===")
        lines.append(chosen_eta_summary.to_string(index=False))
    lines.append("")
    lines.append("=== RQ 2.2 - Prejudice Remover agregado por eta ===")
    lines.append(pr_overall_summary.to_string(index=False) if not pr_overall_summary.empty else "Sem resultados válidos.")
    lines.append("")
    lines.append("=== RQ 2.2 - Prejudice Remover por nível de viés da base de teste ===")
    lines.append(pr_by_bias_summary.to_string(index=False) if not pr_by_bias_summary.empty else "Sem resultados válidos.")
    lines.append("")
    lines.append("=== Comparação direta: baseline vs Prejudice Remover ===")
    lines.append(baseline_vs_pr.to_string(index=False) if not baseline_vs_pr.empty else "Sem comparação disponível.")
    lines.append("")
    lines.append("=== Diagnóstico de erros do Prejudice Remover ===")
    if not eta_validation_df.empty and "error" in eta_validation_df.columns:
        err_val = eta_validation_df["error"].value_counts(dropna=True)
        lines.append("Validação de eta:")
        lines.append(err_val.to_string() if not err_val.empty else "Sem erros.")
    else:
        lines.append("Validação de eta: sem dados.")

    if not pr_test_df.empty and "error" in pr_test_df.columns:
        err_test = pr_test_df["error"].value_counts(dropna=True)
        lines.append("")
        lines.append("Teste PR:")
        lines.append(err_test.to_string() if not err_test.empty else "Sem erros.")

    lines.append("")
    if not baseline_vs_pr.empty:
        lines.append("A comparação baseline vs PR considera apenas resultados válidos do Prejudice Remover.")
    path.write_text("\n".join(lines), encoding="utf-8")


# -------------------------------
# Main
# -------------------------------
def ensure_runtime_dependencies() -> None:
    required = {
        "aif360": "aif360",
        "fairlearn": "fairlearn",
        "openpyxl": "openpyxl",
        "pyarrow": "pyarrow",
    }
    missing = []
    for module_name, pip_name in required.items():
        try:
            __import__(module_name)
        except Exception:
            missing.append(pip_name)

    if missing:
        print(f"[INFO] Instalando dependências ausentes: {', '.join(missing)}")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        print("[INFO] Instalação concluída.")


def main() -> None:
    ensure_runtime_dependencies()

    data_dir = os.environ.get("DATA_DIR", "/content")
    output_dir = os.environ.get("OUTPUT_DIR", "/content/saida_fairness")
    env_target = os.environ.get("TARGET_COL")
    if env_target:
        DEFAULT_COLS["target"] = env_target

    ensure_dir(output_dir)
    datasets: Dict[str, pd.DataFrame] = {}
    cols_map: Dict[str, Dict[str, str]] = {}
    dataset_summary_rows = []

    print(f"[INFO] Procurando bases em: {data_dir}")
    print(f"[INFO] FAST_MODE={FAST_MODE} | USE_CACHE={USE_CACHE} | ETA_GRID={ETA_GRID}")

    for name in DATASET_NAMES:
        path = find_existing_file(data_dir, name)
        if not path:
            print(f"[WARN] Base não encontrada: {name}")
            continue
        print(f"[INFO] Lendo {name}: {path}")
        raw = read_table(path)
        raw = normalize_columns(raw)
        cols = detect_columns(raw)
        cleaned = clean_base(raw)
        cols = detect_columns(cleaned)

        cache_path = Path(output_dir) / f"{name}_enriched_{CACHE_VERSION}.parquet"
        if USE_CACHE and cache_path.exists():
            print(f"[INFO] Usando cache: {cache_path.name}")
            enriched = pd.read_parquet(cache_path)
        else:
            enriched = build_historical_features(cleaned, cols)
            if USE_CACHE:
                enriched.to_parquet(cache_path, index=False)

        datasets[name] = enriched
        cols_map[name] = cols
        dataset_summary_rows.append(summarize_dataset(enriched, name, cols))

    if len(datasets) < 2:
        raise RuntimeError("É preciso ao menos 2 bases para executar o protocolo train-on-one, test-on-others.")

    dataset_summary = pd.DataFrame(dataset_summary_rows).sort_values("dataset")
    dataset_summary.to_csv(Path(output_dir) / "dataset_summary.csv", index=False)

    feature_sets_map = {name: make_feature_sets(df, cols_map[name]) for name, df in datasets.items()}

    print("[INFO] Rodando experimento cross-dataset dos feature sets...")
    cross_results = run_cross_dataset_experiment(datasets, feature_sets_map, cols_map)
    cross_results.to_csv(Path(output_dir) / "cross_dataset_feature_results.csv", index=False)

    feature_summary = aggregate_feature_results(cross_results)
    feature_summary.to_csv(Path(output_dir) / "feature_set_summary.csv", index=False)

    feature_by_bias = aggregate_feature_results_by_test_bias(cross_results)
    feature_by_bias.to_csv(Path(output_dir) / "rq2_1_feature_summary_by_test_bias.csv", index=False)

    stat_tests = run_stat_tests(cross_results)
    stat_tests.to_csv(Path(output_dir) / "rq2_1_stat_tests.csv", index=False)

    bias_test_summary, bias_pair_summary = run_bias_level_analysis(cross_results)
    bias_test_summary.to_csv(Path(output_dir) / "bias_level_test_summary.csv", index=False)
    bias_pair_summary.to_csv(Path(output_dir) / "bias_level_train_test_summary.csv", index=False)

    feature_answers = summarize_feature_set_answers(feature_summary, feature_by_bias)
    feature_answers.to_csv(Path(output_dir) / "rq2_1_feature_answers.csv", index=False)

    chosen_feature_set = select_fixed_feature_set(feature_summary)
    print(f"[INFO] Feature set fixo escolhido para RQ2.2: {chosen_feature_set}")

    print("[INFO] Rodando Prejudice Remover...")
    eta_validation_df, pr_test_df = run_prejudice_remover_experiment(datasets, feature_sets_map, cols_map, chosen_feature_set)
    eta_validation_df.to_csv(Path(output_dir) / "eta_validation_results.csv", index=False)
    pr_test_df.to_csv(Path(output_dir) / "prejudice_remover_test_results.csv", index=False)

    pr_overall_summary, pr_by_bias_summary = aggregate_pr_results(pr_test_df)
    pr_overall_summary.to_csv(Path(output_dir) / "rq2_2_prejudice_remover_overall_summary.csv", index=False)
    pr_by_bias_summary.to_csv(Path(output_dir) / "rq2_2_prejudice_remover_by_test_bias.csv", index=False)

    baseline_vs_pr = build_baseline_vs_pr_table(cross_results, pr_test_df, chosen_feature_set)
    baseline_vs_pr.to_csv(Path(output_dir) / "rq2_2_baseline_vs_prejudice_remover.csv", index=False)

    save_plot_feature_summary(feature_summary, output_dir)
    if not eta_validation_df.empty and {"f1", "fairness_penalty"}.issubset(eta_validation_df.columns):
        save_plot_eta_validation(eta_validation_df, output_dir)
    else:
        print("[WARN] Gráfico de validação de eta não gerado: métricas ausentes.")
        if not eta_validation_df.empty and "error" in eta_validation_df.columns:
            print("[DEBUG] Erros encontrados na validação do Prejudice Remover:")
            print(eta_validation_df[eta_validation_df["error"].notna()][[c for c in ["train_dataset", "eta", "error"] if c in eta_validation_df.columns]].head(20).to_string(index=False))

    write_text_report(
        output_dir=output_dir,
        dataset_summary=dataset_summary,
        feature_summary=feature_summary,
        feature_by_bias=feature_by_bias,
        feature_answers=feature_answers,
        stat_tests=stat_tests,
        chosen_feature_set=chosen_feature_set,
        eta_validation_df=eta_validation_df,
        pr_overall_summary=pr_overall_summary,
        pr_by_bias_summary=pr_by_bias_summary,
        baseline_vs_pr=baseline_vs_pr,
        pr_test_df=pr_test_df,
    )

    meta = {
        "data_dir": data_dir,
        "output_dir": output_dir,
        "datasets_loaded": sorted(list(datasets.keys())),
        "chosen_feature_set": chosen_feature_set,
        "fixed_feature_set_policy": "melhor selection_score entre os conjuntos *_sem_protegido",
        "eta_grid": ETA_GRID,
        "f1_drop_tolerance": F1_DROP_TOLERANCE,
        "fast_mode": FAST_MODE,
        "use_cache": USE_CACHE,
        "pr_validation_max_rows": PR_VALIDATION_MAX_ROWS,
        "max_pr_train_rows": MAX_PR_TRAIN_ROWS,
    }
    (Path(output_dir) / "run_metadata.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

    print("[OK] Execução concluída.")
    print(f"[OK] Saídas salvas em: {output_dir}")


if __name__ == "__main__":
    main()

[INFO] Procurando bases em: /content
[INFO] FAST_MODE=True | USE_CACHE=True | ETA_GRID=[0.0, 1.0, 10.0, 50.0]
[INFO] Lendo gs_v0_5: /content/gs_v0_5.csv
[INFO] Lendo gs_v0_25: /content/gs_v0_25.csv
[INFO] Lendo gs_v0_75: /content/gs_v0_75.csv
[INFO] Lendo gs_v0: /content/gs_v0.csv
[INFO] Lendo gs_v1: /content/gs_v1.csv
[INFO] Lendo pd_v0_5: /content/pd_v0_5.csv
[INFO] Lendo pd_v0_25: /content/pd_v0_25.csv
[INFO] Lendo pd_v0_75: /content/pd_v0_75.csv
[INFO] Lendo pd_v0: /content/pd_v0.csv
[INFO] Lendo pd_v1: /content/pd_v1.csv
[INFO] Rodando experimento cross-dataset dos feature sets...
[INFO] Feature set fixo escolhido para RQ2.2: C_sem_protegido
[INFO] Rodando Prejudice Remover...


pip install 'aif360[inFairness]'
